# Sesión 8 — PCA y t-SNE: ver datos de muchas dimensiones (sin creerle todo al dibujo)

**Introducción al Análisis Avanzado de Datos con Python** · Univalle · Módulo 3 — No supervisados

## Objetivos de hoy
Al terminar la sesión usted podrá:
1. Explicar qué hace PCA (buscar las direcciones de más varianza) y leer la **varianza explicada** para decidir cuántos componentes guardar.
2. Interpretar un **biplot**: qué variables arman cada componente y dónde queda cada fila.
3. Usar **t-SNE** para visualizar datos de muchas dimensiones, sabiendo que conserva vecindades y no distancias.
4. Reconocer cuándo un dibujo de t-SNE **inventa** grupos, tamaños o distancias, y qué se puede concluir honestamente.

## Dónde estamos en el pipeline

```
 ┌────────┐   ┌──────────────────┐   ┌────────┐   ┌────────────┐   ┌────────────────┐
 │ DATOS  │ → │ PREPROCESAMIENTO │ → │ MODELO │ → │ EVALUACIÓN │ → │ INTERPRETACIÓN │
 └────────┘   └──────────────────┘   └────────┘   └────────────┘   └────────────────┘
                ▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲                                   ▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲
   HOY: resumir muchas columnas en pocas (preprocesamiento) y dibujarlas (interpretación)… con cuidado.
```

🔍 **Predice** · ✏️ **Completa** (andamiaje medio-bajo) · 💥 **Trampa** · 🖼️ **Fichas** · ⚡ **Mini-reto**.

> **Datos:** los **pingüinos** de la Sesión 7 (4 medidas) y los **dígitos escritos a mano** que trae scikit-learn (1 797 imágenes de 8×8 píxeles = 64 columnas; Alpaydin y Kaynak, UCI ML Repository). No hay que descargar nada nuevo.


## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

pd.set_option("display.max_columns", 40)
REPO = "https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/"
ARCHIVOS = {"penguins.csv": "datos/penguins.csv", "figuras.py": "tools/figuras.py"}

def asegurar_datos(archivo, url_repo):
    """Deja un archivo en el disco de Colab: lo baja del repositorio del curso si aún no está."""
    from pathlib import Path
    import requests
    if Path(archivo).exists():
        return True
    try:
        r = requests.get(url_repo, timeout=120)
        if r.ok:
            Path(archivo).write_bytes(r.content)
            return True
    except Exception:
        pass
    print(f"⚠️ No pude bajar {archivo}. Use el Plan B (celda siguiente).")
    return False

for archivo, ruta in ARCHIVOS.items():
    asegurar_datos(archivo, REPO + ruta)
import figuras as fg
print("Listo:", ", ".join(ARCHIVOS))

<details><summary><b>Plan B</b> — si la celda anterior avisó que no pudo bajar algún archivo</summary>

La celda siguiente revisa sola qué falta y solo en ese caso le pide subirlo.
</details>

In [ ]:
# Plan B: solo actúa si falta algún archivo. Con "Ejecutar todo" no se detiene.
from pathlib import Path
faltan = [a for a in ARCHIVOS if not Path(a).exists()]
if not faltan:
    print("✓ Todo disponible: no hace falta el Plan B.")
else:
    from google.colab import files
    print("⚠️ Faltan:", faltan, "→ súbalos")
    files.upload()

In [ ]:
fg.pipeline(["preprocesamiento", "interpretación"], modulo=3, subtitulo="HOY: resumir muchas columnas en pocas, y leer el dibujo con honestidad");

## 1. El problema: no vemos más de 2 dimensiones

En la Sesión 7 dibujamos a los pingüinos con 2 de sus 4 medidas. Para verlas todas hay que mirar **todas las parejas**:

In [ ]:
pinguinos = pd.read_csv("penguins.csv")
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
pinguinos = pinguinos.dropna(subset=medidas).reset_index(drop=True)
fg.todas_las_parejas(pinguinos, medidas, color_col="species");

Seis gráficas para 4 variables ya es incómodo; con los 64 píxeles de una imagen serían 2 016. La **reducción de dimensionalidad** busca pocas columnas nuevas que resuman muchas, perdiendo lo menos posible.

## 2. PCA: la dirección en la que más se esparcen los datos

**PCA** (*Principal Component Analysis*, análisis de componentes principales) busca la dirección en la que los puntos están **más esparcidos** y la llama **PC1** (primer componente principal). Después busca la siguiente, perpendicular a la primera (**PC2**), y así. Proyectar sobre PC1 es como mirar la "sombra" de los datos sobre esa línea:

In [ ]:
fg.pca_proyeccion();

### 📐 Fundamento
Cada componente es una **suma ponderada** de las variables (escaladas):

$$\text{PC1} = w_1 x_1 + w_2 x_2 + \dots + w_p x_p$$

Los pesos $w$ se eligen para que PC1 tenga la **máxima varianza** posible (con $\sum w_j^2 = 1$, para que no "haga trampa" agrandando los pesos). PC2 hace lo mismo con la condición de no repetir la información de PC1, y así sucesivamente.

La **varianza explicada** del componente $k$ es la fracción de la variación total que recoge:

$$\text{varianza explicada}_k = \frac{\lambda_k}{\lambda_1 + \lambda_2 + \dots + \lambda_p}$$

donde $\lambda_k$ es la varianza de los datos sobre el componente $k$. Si PC1 y PC2 suman, digamos, 90 %, el dibujo en 2D es casi fiel.

<details><summary><b>Para quien quiera más:</b> de dónde salen los pesos</summary>

Los pesos de cada componente son los **vectores propios** de la matriz de covarianza de los datos (o de correlación, si se escaló), y las $\lambda_k$ son sus **valores propios**. No hay iteraciones ni semilla: es álgebra lineal, siempre da lo mismo (salvo el signo de cada componente, que es arbitrario). Por eso PCA es una referencia confiable para comparar con métodos más vistosos.
</details>

![Dentro del .fit() — PCA y t-SNE](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S08/img/dentro_del_fit_pca_tsne.png)

## 3. PCA con los pingüinos

### 🔍 Predice
Antes de escalar, la lección de la Sesión 7. Si hacemos PCA con las 4 medidas **tal como vienen** (gramos y milímetros), ¿qué porcentaje de la varianza recogerá PC1?

**Tu predicción:** ____ %

In [ ]:
from sklearn.decomposition import PCA

pca_crudo = PCA().fit(pinguinos[medidas])
print("varianza explicada sin escalar:", (pca_crudo.explained_variance_ratio_ * 100).round(3))
print("pesos de PC1:", {m: round(float(w), 3) for m, w in zip(medidas, pca_crudo.components_[0])})

### 🔍 Predice
Con las medidas **escaladas**, ¿qué porcentaje de la varianza recogerán **juntos** PC1 y PC2?

**Tu predicción:** ____ %

### ✏️ Completa — escale y ajuste PCA
Escale con `StandardScaler` y ajuste `PCA()` (sin `n_components`: así calcula los 4 y podemos ver todos).

In [ ]:
from sklearn.preprocessing import StandardScaler

Z = ...     # TODO: las medidas escaladas con StandardScaler
pca = ...   # TODO: PCA() ajustado con Z
fg.pca_varianza(pca);

### ¿Qué significa cada componente?
Los pesos están en `pca.components_` (una fila por componente). El **biplot** los dibuja como flechas encima de los pingüinos proyectados:

In [ ]:
pd.DataFrame(pca.components_[:2], columns=medidas, index=["PC1", "PC2"]).round(2)

In [ ]:
scores = pca.transform(Z)
fg.biplot(scores, pca.components_, medidas, etiquetas=pinguinos["species"], var_exp=pca.explained_variance_ratio_);

### Una fila entra, un número sale
El valor de PC1 de un pingüino es la suma de sus medidas escaladas por los pesos, como una predicción de la regresión de la Sesión 4:

In [ ]:
fg.pca_una_fila(Z[0], pca.components_[0], medidas, nombre="PC1", real=pinguinos.loc[0, "species"]);

## 4. Ahora sí, muchas dimensiones: 64 píxeles

Los dígitos de scikit-learn son 1 797 imágenes de 8×8 píxeles. Cada imagen es una fila de **64 columnas**.

In [ ]:
from sklearn.datasets import load_digits

X_dig, y_dig = load_digits(return_X_y=True)
print(X_dig.shape)
fg.digitos_muestra(X_dig, y_dig);

### 🔍 Predice
Con 64 columnas, ¿qué porcentaje de la varianza recogerán los **2 primeros** componentes? ¿Cuántos componentes harán falta para llegar al 90 %?

**Tu predicción:** 2 componentes → ____ % · para 90 % hacen falta ____

### ✏️ Completa — PCA sobre los dígitos
Aquí todas las columnas están en la misma unidad (intensidad de 0 a 16), así que no hace falta escalar.

In [ ]:
pca_dig = ...   # TODO: PCA() ajustado con X_dig
print(f"2 componentes: {pca_dig.explained_variance_ratio_[:2].sum():.1%}")
fg.pca_varianza(pca_dig, marcar=0.9, titulo="Dígitos: 64 componentes");

## 5. t-SNE: dibujar vecindades

**t-SNE** (*t-distributed Stochastic Neighbor Embedding*) no busca direcciones: busca acomodar los puntos en 2D de modo que **cada punto quede cerca de los mismos vecinos** que tenía en 64D. No le importa conservar distancias grandes, solo quién es vecino de quién.

Su perilla principal es la **perplexity**: más o menos, **cuántos vecinos** mira cada punto (valores típicos de 5 a 50).

### ✏️ Completa — t-SNE de los dígitos
`TSNE(n_components=2, perplexity=30, random_state=42)` y `.fit_transform(X_dig)`. Tarda unos segundos: está acomodando 1 797 puntos uno por uno.

In [ ]:
from sklearn.manifold import TSNE

T_dig = ...   # TODO: TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_dig)
fg.pca_vs_tsne(X_dig, y_dig, pca_dig.transform(X_dig)[:, :2], T_dig);

Espectacular: t-SNE separa casi todos los dígitos sin haber visto las etiquetas. ¿Entonces t-SNE "encontró" 10 grupos?

<details><summary><b>Para quien quiera más:</b> qué minimiza t-SNE</summary>

Para cada par de puntos calcula una probabilidad de "ser vecinos" en 64D, $p_{ij}$ (una campana gaussiana cuyo ancho se ajusta a la perplexity), y otra en 2D, $q_{ij}$ (una distribución *t* de Student, de colas largas). Mueve los puntos 2D con **descenso de gradiente** (Sesión 4) para minimizar la divergencia de Kullback–Leibler:

$$\text{KL} = \sum_{i \ne j} p_{ij} \log \frac{p_{ij}}{q_{ij}}$$

Esa fórmula castiga mucho separar a dos vecinos verdaderos y casi nada juntar a dos puntos lejanos. Por eso conserva vecindades y no distancias. Y como arranca de posiciones al azar y la función tiene muchos mínimos locales (como K-means), el resultado depende de la semilla.
</details>

## 6. 💥 ¿Le creemos al dibujo?

### 🔍 Predice
Si cambiamos la perplexity a **2** o a **100**, ¿el dibujo se parecerá al anterior?

**Tu predicción:** ____

Para que corra rápido, usamos 900 de los 1 797 dígitos.

In [ ]:
muestra = np.random.default_rng(0).choice(len(X_dig), 900, replace=False)
fg.tsne_perplexity(X_dig[muestra], y_dig[muestra], perplexities=(2, 5, 30, 100));

In [ ]:
fg.tsne_semillas(X_dig[muestra], y_dig[muestra], semillas=(0, 1, 2, 3));

Con perplexity 2 aparecen islotes y astillas que **no existen**; con 100 los grupos se acercan. Y con cada semilla, el mismo dígito cae en otro lugar y con otros vecinos. Lo que sí se repite: **qué** grupos hay. Lo que no: **dónde** quedan y **qué tan lejos** están.

### 🔍 Predice
Ahora datos **sin ningún grupo**: 500 puntos al azar, uniformes, en un cuadrado. ¿Qué dibujará t-SNE?

**Tu predicción:** ____

In [ ]:
fg.tsne_ruido(n=500, perplexities=(2, 5, 50));

Y una última prueba: tres grupos en 10 dimensiones. **A** es disperso, **B** es diez veces más apretado y está pegado a A, **C** está muy lejos. PCA los muestra como son; ¿y t-SNE?

In [ ]:
fg.tsne_tamanos();

## 7. Entonces, ¿para qué sirve cada uno?

| | PCA | t-SNE |
|---|---|---|
| Conserva | la varianza global (distancias grandes) | las vecindades (quién está cerca de quién) |
| Ejes | interpretables (biplot: pesos de cada variable) | sin significado |
| Filas nuevas | sí: `.transform()` | no: hay que recalcular todo |
| Repetible | siempre da lo mismo | depende de semilla y perplexity |
| Velocidad | instantáneo | lento con muchas filas |
| Úselo para | comprimir columnas antes de otro modelo; entender qué variables van juntas; un primer dibujo honesto | explorar si hay estructura local; **nunca** como prueba única de grupos |

Regla práctica: **primero PCA**, y si se ve borroso, t-SNE con varias perplexities y semillas, comparando siempre con algo que no se usó para dibujar.

## ⚡ Mini-reto (opcional)

Haga t-SNE de los pingüinos **escalados** (`Z`) con perplexity 5, 30 y 100, coloreando por especie. ¿Separa mejor que el biplot de PCA? ¿Qué le pasa a Gentoo con perplexity 5? ¿Algún dibujo sugiere más de 3 grupos? ¿Le creería?

In [ ]:
# Su análisis aquí
# for p in (5, 30, 100):
#     T = TSNE(n_components=2, perplexity=p, random_state=0).fit_transform(Z)
#     ...

## 8. Cierre — qué cambió hoy en el pipeline

| Etapa | Hoy (Sesión 8) |
|---|---|
| Preprocesamiento | PCA como **compresión**: 64 columnas → 21 con el 90 % de la información. Escalar antes (si no, gana la unidad más grande) |
| Modelo | PCA (fórmula cerrada) y t-SNE (descenso de gradiente, semilla, perplexity) |
| Interpretación | biplot: qué variables arman cada eje; t-SNE: solo vecindades, nunca tamaños ni distancias |

**Dentro del `.fit()` — los no supervisados:**

| | K-means | DBSCAN | PCA | t-SNE |
|---|---|---|---|---|
| Para qué | agrupar | agrupar (densidad) | reducir dimensiones | dibujar vecindades |
| Qué optimiza | minimiza inercia | nada | maximiza varianza proyectada | minimiza KL entre vecindades |
| Cómo lo busca | iterativo, mínimo local | encadena vecinos | álgebra lineal (cerrada) | descenso de gradiente, mínimo local |
| ¿Depende de la semilla? | sí (`n_init`) | no | no | **sí** |
| Perilla | `k` | `eps`, `min_samples` | `n_components` | `perplexity` |
| ¿Escalar? | sí | sí | sí | sí |

**Para la Sesión 9:** juntamos todo sobre datos de Colombia: indicadores de pobreza de los municipios (Censo 2018). PCA para entenderlos, K-means para agruparlos, un **mapa** para verlos y el perfil honesto de cada grupo. Y se presenta la entrega **E3** sobre su propio dataset.